# Incremental approximate BRs for the 69-claim overnight CFR+ run

This notebook evaluates the 6-hour `r6_s4_h4_hp2ptfhq_ss` neural CFR+ overnight run with action-conditioned fitted-return best responders.

Workflow:

1. Train one responder per hourly policy snapshot for 5 measured responder minutes, plot, then continue the same responders to 10, 15, and 20 minutes.
2. Free the hourly responder trainers from GPU memory, keeping their logged 20-minute results on disk.
3. Repeat the same 5/10/15/20-minute incremental responder schedule for the half-hour snapshots.
4. Half-hour plots include the already-completed 20-minute hourly responder curve for reference.

The responder trainers are intentionally kept alive between cells during each group, so run cells in order. If the kernel dies, start that group again; existing logs are preserved but the BR replay state is not serialized by this notebook.

In [ ]:
import gc
import json
from pathlib import Path
import shutil
import sys
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate the liars_poker repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_fitted_return_action_conditioned import ActionConditionedFittedReturnBRTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.approx_br import evaluate_approximate_br
from liars_poker.serialization import load_policy, save_policy

assert torch.cuda.is_available(), 'These responder runs are intended for CUDA.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 69

# Default from the completed overnight-run summary. If this path is absent, the next cell
# tries to discover the newest matching run under artifacts/cfr_plus_69_claim_overnight.
RUN_DIR = Path('/root/liars_poker/artifacts/cfr_plus_69_claim_overnight/r6_s4_h4_hp2ptfhq_ss___20260625-044644')

OUTPUT_ROOT = None  # set after RUN_DIR is resolved
HOURLY_MINUTES = [60, 120, 180, 240, 300, 360]
HALF_HOUR_MINUTES = [30, 90, 150, 210, 270, 330]
RESPONDER_TOTAL_MINUTES = [5, 10, 15, 20]
RESPONDER_SEED = 17

BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024
BR_EVAL_EPISODES_PER_ROLE = 200_000
BR_EVAL_ROLLOUT_BATCH_SIZE = 8192
PRINT_EVERY_RESPONDER_MINUTES = 1.0

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
    'seed': RESPONDER_SEED,
}

print('repository:', REPO_ROOT)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, tuple):
        return list(obj)
    raise TypeError(f'Object of type {type(obj).__name__} is not JSON serializable')


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, default=json_default, sort_keys=True) + '\n')


def matching_run_candidates() -> pd.DataFrame:
    root = REPO_ROOT / 'artifacts' / 'cfr_plus_69_claim_overnight'
    wanted = json.loads(spec.to_json())
    rows = []
    if not root.exists():
        return pd.DataFrame(rows)
    for run_dir in root.iterdir():
        manifest_path = run_dir / 'manifest.json'
        snapshot_root = run_dir / 'policy_snapshots'
        if not run_dir.is_dir() or not manifest_path.exists() or not snapshot_root.exists():
            continue
        manifest = read_json(manifest_path)
        raw_spec = manifest.get('spec')
        try:
            run_spec = json.loads(raw_spec) if isinstance(raw_spec, str) else raw_spec
        except (TypeError, json.JSONDecodeError):
            continue
        if run_spec != wanted:
            continue
        snapshots = [p for p in snapshot_root.iterdir() if p.is_dir() and (p / 'metadata.json').exists()]
        summary_path = run_dir / 'summary.json'
        summary = read_json(summary_path) if summary_path.exists() else {}
        rows.append({
            'run_dir': run_dir,
            'snapshots': len(snapshots),
            'summary_training_min': summary.get('measured_training_min'),
            'summary_status': summary.get('status'),
            'mtime': run_dir.stat().st_mtime,
        })
    return pd.DataFrame(rows).sort_values(['mtime'], ascending=False).reset_index(drop=True)


if not RUN_DIR.exists():
    candidates = matching_run_candidates()
    if candidates.empty:
        raise FileNotFoundError(f'Default RUN_DIR does not exist and no matching local run was found: {RUN_DIR}')
    RUN_DIR = Path(candidates.iloc[0]['run_dir'])
    print('Using newest discovered run:', RUN_DIR)
else:
    print('Using configured run:', RUN_DIR)

OUTPUT_ROOT = RUN_DIR / 'approx_br_incremental_5m'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('output root:', OUTPUT_ROOT)
print('free disk GiB at run dir:', round(shutil.disk_usage(RUN_DIR).free / 2**30, 2))

In [ ]:
def snapshot_policy_dir(target_minute: int) -> Path:
    label = f'{int(target_minute):04d}m'
    path = RUN_DIR / 'policy_snapshots' / label
    if path.exists() and (path / 'metadata.json').exists():
        return path
    if int(target_minute) == 360:
        final_path = RUN_DIR / 'final_policy'
        if final_path.exists() and (final_path / 'metadata.json').exists():
            return final_path
    raise FileNotFoundError(f'Missing policy snapshot for target minute {target_minute}: expected {path}')


def snapshot_metadata(policy_dir: Path) -> dict[str, Any]:
    snapshot_path = policy_dir / 'snapshot.json'
    if snapshot_path.exists():
        return read_json(snapshot_path)
    summary_path = RUN_DIR / 'summary.json'
    summary = read_json(summary_path) if summary_path.exists() else {}
    return {
        'measured_training_min': summary.get('measured_training_min'),
        'measured_training_s': summary.get('measured_training_s'),
        'iteration': summary.get('iteration'),
        'path': str(policy_dir),
    }


def snapshot_table(minutes: list[int], group: str) -> pd.DataFrame:
    rows = []
    for minute in minutes:
        policy_dir = snapshot_policy_dir(minute)
        meta = snapshot_metadata(policy_dir)
        _, loaded_spec = load_policy(str(policy_dir))
        if loaded_spec != spec:
            raise ValueError(f'Spec mismatch for {policy_dir}: {loaded_spec} != {spec}')
        rows.append({
            'group': group,
            'target minute': int(minute),
            'policy_dir': policy_dir,
            'snapshot training min': float(meta.get('measured_training_min', minute) or minute),
            'snapshot iteration': int(meta.get('iteration', -1) or -1),
        })
    return pd.DataFrame(rows)

hourly_targets = snapshot_table(HOURLY_MINUTES, 'hourly')
half_hour_targets = snapshot_table(HALF_HOUR_MINUTES, 'half_hour')
display(hourly_targets[['group', 'target minute', 'snapshot training min', 'snapshot iteration', 'policy_dir']])
display(half_hour_targets[['group', 'target minute', 'snapshot training min', 'snapshot iteration', 'policy_dir']])

In [ ]:
def target_dir(group: str, target_minute: int) -> Path:
    return OUTPUT_ROOT / group / f'snapshot_{int(target_minute):04d}m'


def make_state(row: pd.Series, *, seed: int) -> dict[str, Any]:
    policy, loaded_spec = load_policy(str(row['policy_dir']))
    if loaded_spec != spec:
        raise ValueError(f'Spec mismatch for {row["policy_dir"]}')
    kwargs = dict(BR_TRAINER_KWARGS)
    kwargs['seed'] = int(seed)
    trainer = ActionConditionedFittedReturnBRTrainer(loaded_spec, policy, **kwargs)
    out_dir = target_dir(str(row['group']), int(row['target minute']))
    out_dir.mkdir(parents=True, exist_ok=True)
    metadata = {
        'group': str(row['group']),
        'target_minute': int(row['target minute']),
        'snapshot_training_min': float(row['snapshot training min']),
        'snapshot_iteration': int(row['snapshot iteration']),
        'policy_dir': str(row['policy_dir']),
        'responder_seed': int(seed),
        'trainer_kwargs': {k: (str(v) if k == 'device' else v) for k, v in BR_TRAINER_KWARGS.items()},
    }
    (out_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2, default=json_default), encoding='utf-8')
    return {
        **metadata,
        'trainer': trainer,
        'out_dir': out_dir,
        'measured_responder_s': 0.0,
        'best_p_first': -np.inf,
        'best_p_second': -np.inf,
        'best_p_first_lcb': -np.inf,
        'best_p_second_lcb': -np.inf,
    }


def make_states(targets: pd.DataFrame, *, seed: int = RESPONDER_SEED) -> dict[int, dict[str, Any]]:
    states = {}
    for _, row in targets.iterrows():
        minute = int(row['target minute'])
        print(f'loading responder target {row["group"]} {minute}m')
        states[minute] = make_state(row, seed=seed)
    print('free / total GPU GiB after loading responders:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))
    return states


def flatten_training_record(state: dict[str, Any], record: dict[str, Any], iteration_s: float) -> dict[str, Any]:
    roles = record.get('roles', [])
    role_rows = {}
    for role_index, role_record in enumerate(roles):
        prefix = f'role_{role_index + 1}'
        for key, value in role_record.items():
            role_rows[f'{prefix}_{key}'] = value
    return {
        'group': state['group'],
        'target_minute': state['target_minute'],
        'snapshot_training_min': state['snapshot_training_min'],
        'snapshot_iteration': state['snapshot_iteration'],
        'responder_seed': state['responder_seed'],
        'iteration': int(record.get('iter', -1)),
        'iteration_wall_s': float(iteration_s),
        'measured_responder_s': float(state['measured_responder_s']),
        **role_rows,
    }


def train_state_to(state: dict[str, Any], target_total_min: float) -> None:
    trainer = state['trainer']
    target_s = 60.0 * float(target_total_min)
    next_print_s = (int(state['measured_responder_s'] // (60.0 * PRINT_EVERY_RESPONDER_MINUTES)) + 1) * 60.0 * PRINT_EVERY_RESPONDER_MINUTES
    while state['measured_responder_s'] < target_s:
        start = time.perf_counter()
        record = trainer.run_iteration(
            episodes_per_role=BR_EPISODES_PER_ROLE,
            rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
        )
        iteration_s = time.perf_counter() - start
        state['measured_responder_s'] += iteration_s
        row = flatten_training_record(state, record, iteration_s)
        append_jsonl(state['out_dir'] / 'training.jsonl', row)
        append_jsonl(OUTPUT_ROOT / 'training.jsonl', row)
        if state['measured_responder_s'] >= next_print_s:
            print(
                f"  {state['group']} {state['target_minute']:04d}m: "
                f"responder={state['measured_responder_s'] / 60:.1f}m "
                f"iter={trainer.iteration} iter_s={iteration_s:.3f}"
            )
            next_print_s += 60.0 * PRINT_EVERY_RESPONDER_MINUTES


def evaluate_state(state: dict[str, Any], target_total_min: float) -> dict[str, Any]:
    trainer = state['trainer']
    metrics = evaluate_approximate_br(
        trainer,
        episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
        rollout_batch_size=BR_EVAL_ROLLOUT_BATCH_SIZE,
    )
    state['best_p_first'] = max(float(state['best_p_first']), float(metrics['p_first']))
    state['best_p_second'] = max(float(state['best_p_second']), float(metrics['p_second']))
    state['best_p_first_lcb'] = max(float(state['best_p_first_lcb']), float(metrics['p_first_lcb']))
    state['best_p_second_lcb'] = max(float(state['best_p_second_lcb']), float(metrics['p_second_lcb']))
    row = {
        'group': state['group'],
        'target_minute': state['target_minute'],
        'snapshot_training_min': state['snapshot_training_min'],
        'snapshot_iteration': state['snapshot_iteration'],
        'responder_seed': state['responder_seed'],
        'responder_target_total_min': float(target_total_min),
        'responder_training_min': float(state['measured_responder_s']) / 60.0,
        'responder_iteration': int(trainer.iteration),
        **metrics,
        'best_p_first': float(state['best_p_first']),
        'best_p_second': float(state['best_p_second']),
        'best_p_first_lcb': float(state['best_p_first_lcb']),
        'best_p_second_lcb': float(state['best_p_second_lcb']),
        'best_discovered_estimate': float(state['best_p_first'] + state['best_p_second'] - 1.0),
        'conservative_lower_bound': float(state['best_p_first_lcb'] + state['best_p_second_lcb'] - 1.0),
        'out_dir': str(state['out_dir']),
    }
    append_jsonl(state['out_dir'] / 'evaluations.jsonl', row)
    append_jsonl(OUTPUT_ROOT / 'evaluations.jsonl', row)
    if float(target_total_min) >= max(RESPONDER_TOTAL_MINUTES):
        save_policy(trainer.policy(), str(state['out_dir'] / 'final_responder_policy'))
    return row


def run_responder_round(states: dict[int, dict[str, Any]], target_total_min: float) -> pd.DataFrame:
    rows = []
    for minute in sorted(states):
        state = states[minute]
        print(f"\n[{state['group']} {minute:04d}m] training responder to {target_total_min:.0f} total minutes")
        train_state_to(state, target_total_min)
        row = evaluate_state(state, target_total_min)
        rows.append(row)
        print(
            f"  estimate={row['best_discovered_estimate']:.5f} "
            f"LCB={row['conservative_lower_bound']:.5f} "
            f"p=({row['best_p_first']:.5f}, {row['best_p_second']:.5f})"
        )
    return pd.DataFrame(rows)


def release_states(states: dict[int, dict[str, Any]]) -> None:
    for state in states.values():
        state.pop('trainer', None)
    states.clear()
    gc.collect()
    torch.cuda.empty_cache()
    print('free / total GPU GiB after release:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
def load_all_evaluations() -> pd.DataFrame:
    rows = read_jsonl(OUTPUT_ROOT / 'evaluations.jsonl')
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(
        subset=['group', 'target_minute', 'responder_seed', 'responder_target_total_min'],
        keep='last',
    )
    return df.sort_values(['group', 'target_minute', 'responder_target_total_min']).reset_index(drop=True)


def latest_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    idx = df.sort_values('responder_target_total_min').groupby(['group', 'target_minute']).tail(1).index
    return df.loc[idx].sort_values(['group', 'snapshot_training_min'])


def plot_progress(focus_group: str) -> pd.DataFrame:
    df = load_all_evaluations()
    if df.empty:
        print('No evaluations have been logged yet.')
        return df

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    focus = df[df['group'] == focus_group].copy()
    if focus.empty:
        print(f'No evaluations for group {focus_group!r} yet.')
    else:
        for target_total, group in focus.groupby('responder_target_total_min'):
            group = group.sort_values('snapshot_training_min')
            axes[0].plot(
                group['snapshot_training_min'],
                group['best_discovered_estimate'],
                marker='o',
                label=f'{target_total:g}m BR',
            )
            axes[1].plot(
                group['snapshot_training_min'],
                group['conservative_lower_bound'],
                marker='o',
                label=f'{target_total:g}m BR',
            )

        for snapshot_min, group in focus.groupby('snapshot_training_min'):
            group = group.sort_values('responder_training_min')
            axes[2].plot(
                group['responder_training_min'],
                group['best_discovered_estimate'],
                marker='o',
                alpha=0.75,
                label=f'{snapshot_min:.0f}m CFR+',
            )

    hourly_reference = df[
        (df['group'] == 'hourly')
        & (df['responder_target_total_min'] == max(RESPONDER_TOTAL_MINUTES))
    ].sort_values('snapshot_training_min')
    if focus_group != 'hourly' and not hourly_reference.empty:
        axes[0].plot(
            hourly_reference['snapshot_training_min'],
            hourly_reference['best_discovered_estimate'],
            color='black',
            marker='s',
            linestyle='--',
            linewidth=2.0,
            label='hourly 20m BR reference',
        )
        axes[1].plot(
            hourly_reference['snapshot_training_min'],
            hourly_reference['conservative_lower_bound'],
            color='black',
            marker='s',
            linestyle='--',
            linewidth=2.0,
            label='hourly 20m BR reference',
        )

    axes[0].set(
        title=f'{focus_group}: discovered exploitability over CFR+ training',
        xlabel='CFR+ snapshot training minutes',
        ylabel='Best discovered estimate',
    )
    axes[1].set(
        title=f'{focus_group}: conservative MC lower bound',
        xlabel='CFR+ snapshot training minutes',
        ylabel='Lower-bound estimate',
    )
    axes[2].set(
        title=f'{focus_group}: responder compute curves',
        xlabel='Responder training minutes',
        ylabel='Best discovered estimate',
    )
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    summary = latest_summary(df)
    display(summary[[
        'group',
        'target_minute',
        'snapshot_training_min',
        'snapshot_iteration',
        'responder_target_total_min',
        'responder_training_min',
        'best_p_first',
        'best_p_second',
        'best_discovered_estimate',
        'conservative_lower_bound',
    ]].sort_values(['group', 'snapshot_training_min']))
    return df

## Hourly snapshots

Run these four cells in order. Each cell continues the same hourly responders by another five measured responder minutes and then replots the hourly curves.

In [ ]:
hourly_states = make_states(hourly_targets, seed=RESPONDER_SEED)
round_5_hourly = run_responder_round(hourly_states, 5)
plot_progress('hourly')

In [ ]:
round_10_hourly = run_responder_round(hourly_states, 10)
plot_progress('hourly')

In [ ]:
round_15_hourly = run_responder_round(hourly_states, 15)
plot_progress('hourly')

In [ ]:
round_20_hourly = run_responder_round(hourly_states, 20)
plot_progress('hourly')

In [ ]:
# Free the hourly BR trainers before starting the half-hour group. The hourly
# 20-minute results stay on disk and will still be included in later plots.
release_states(hourly_states)

## Half-hour snapshots

These cells repeat the same incremental schedule for 30, 90, 150, 210, 270, and 330 minute CFR+ snapshots. The plots include the hourly 20-minute BR results as a black dashed reference curve.

In [ ]:
half_hour_states = make_states(half_hour_targets, seed=RESPONDER_SEED)
round_5_half_hour = run_responder_round(half_hour_states, 5)
plot_progress('half_hour')

In [ ]:
round_10_half_hour = run_responder_round(half_hour_states, 10)
plot_progress('half_hour')

In [ ]:
round_15_half_hour = run_responder_round(half_hour_states, 15)
plot_progress('half_hour')

In [ ]:
round_20_half_hour = run_responder_round(half_hour_states, 20)
plot_progress('half_hour')

In [ ]:
release_states(half_hour_states)

## Final combined view

This cell is safe to rerun any time after some evaluations have been logged.

In [ ]:
df = load_all_evaluations()
if df.empty:
    raise RuntimeError('No evaluation rows found yet.')

final_rows = df.sort_values('responder_target_total_min').groupby(['group', 'target_minute']).tail(1).copy()
final_rows = final_rows.sort_values('snapshot_training_min')

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for group_name, group in final_rows.groupby('group'):
    group = group.sort_values('snapshot_training_min')
    axes[0].plot(
        group['snapshot_training_min'],
        group['best_discovered_estimate'],
        marker='o',
        label=group_name,
    )
    axes[1].plot(
        group['snapshot_training_min'],
        group['conservative_lower_bound'],
        marker='o',
        label=group_name,
    )
axes[0].set(title='Final available BR estimate by snapshot', xlabel='CFR+ training minutes', ylabel='Best discovered estimate')
axes[1].set(title='Final available conservative lower bound', xlabel='CFR+ training minutes', ylabel='Conservative lower bound')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

display(final_rows[[
    'group',
    'target_minute',
    'snapshot_training_min',
    'snapshot_iteration',
    'responder_target_total_min',
    'responder_training_min',
    'best_p_first',
    'best_p_second',
    'best_discovered_estimate',
    'conservative_lower_bound',
    'out_dir',
]].sort_values(['snapshot_training_min', 'group']))

csv_path = OUTPUT_ROOT / 'incremental_br_summary.csv'
final_rows.to_csv(csv_path, index=False)
print('wrote', csv_path)